Natural Language Processing & Text Analytics - Session 07 - March 30 2022

## Exercise 7.0* Topic Modeling Using Gensim

Follow this exercise to go through the setup of an LDA model. Here is an article on LDA, if you need some clarifying examples: https://towardsdatascience.com/latent-dirichlet-allocation-lda-9d1cd064ffa2

**The Data**

The data set we’ll use is a list of over one million news headlines published over a period of 15 years and can be downloaded from [Kaggle](https://www.kaggle.com/therohk/million-headlines/data?login=true#abcnews-date-text.csv).

In [8]:
# this is only needed when you are running the codes on Google Colab.
#from google.colab import files
#uploaded = files.upload()

#delete the uploaded file
#!rm 'chatbot.txt'

In [9]:
import pandas as pd

data = pd.read_csv('abcnews-date-text.csv')
data_text = data[['headline_text']]
data_text['index'] = data_text.index
documents = data_text

In [10]:
len(documents)

1226258

In [11]:
documents[:5]

,headline_text,index
0,aba decides against community broadcasting lic...,0
1,act fire witnesses must be aware of defamation,1
2,a g calls for infrastructure protection summit,2
3,air nz staff in aust strike for pay rise,3
4,air nz strike to affect australian travellers,4


**Data Preprocessing**

We will perform the following steps:

Tokenization: Split the text into sentences and the sentences into words. 

Lowercase the words and remove punctuation.

Words that have fewer than 3 characters are removed.

All stopwords are removed.

Words are lemmatized — words in third person are changed to first person and verbs in past and future tenses are changed into present.

Words are stemmed — words are reduced to their root form.


In [12]:
#install the package paramiko
!pip install paramiko

Defaulting to user installation because normal site-packages is not writeable


In [13]:
!pip install gensim
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(2018)

Defaulting to user installation because normal site-packages is not writeable


/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [14]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/johannuchelwildt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Lemmatize example

In [15]:
nltk.download('omw-1.4')
print(WordNetLemmatizer().lemmatize('went', pos='v'))

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/johannuchelwildt/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


go


### Stemmer Example

In [16]:
stemmer = SnowballStemmer('english')
original_words = ['caresses', 'flies', 'dies', 'mules', 'denied','died', 'agreed', 'owned', 
           'humbled', 'sized','meeting', 'stating', 'siezing', 'itemization','sensational', 
           'traditional', 'reference', 'colonizer','plotted']
singles = [stemmer.stem(plural) for plural in original_words]
pd.DataFrame(data = {'original word': original_words, 'stemmed': singles})

,original word,stemmed
0,caresses,caress
1,flies,fli
2,dies,die
3,mules,mule
4,denied,deni
5,died,die
6,agreed,agre
7,owned,own
8,humbled,humbl
9,sized,size


Write a function to perform lemmatize and stem preprocessing steps on the data set.

In [17]:
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
    return result

Select a document to preview after preprocessing.

In [18]:
print(documents.shape)

(1226258, 2)


In [19]:
doc_sample = documents[documents['index'] == 4310].values[0][0]

print('original document: ')
words = []
for word in doc_sample.split(' '):
    words.append(word)
print(words)
print('\n\n tokenized and lemmatized document: ')   
print(preprocess(doc_sample))

original document: 
['ratepayers', 'group', 'wants', 'compulsory', 'local', 'govt', 'voting']


 tokenized and lemmatized document: 
['ratepay', 'group', 'want', 'compulsori', 'local', 'govt', 'vote']


Preprocess the headline text, saving the results as ‘processed_docs’

In [20]:

processed_docs = documents['headline_text'][:10000].map(preprocess)

In [21]:
processed_docs[:10]

0            [decid, communiti, broadcast, licenc]
1                               [wit, awar, defam]
2           [call, infrastructur, protect, summit]
3                      [staff, aust, strike, rise]
4             [strike, affect, australian, travel]
5               [ambiti, olsson, win, tripl, jump]
6           [antic, delight, record, break, barca]
7    [aussi, qualifi, stosur, wast, memphi, match]
8            [aust, address, secur, council, iraq]
9                         [australia, lock, timet]
Name: headline_text, dtype: object

## **Bag of Words on the Data set**

Create a dictionary from ‘processed_docs’ containing the number of times a word appears in the training set.

In [22]:
dictionary = gensim.corpora.Dictionary(processed_docs)

In [23]:
print(len(dictionary.iteritems()))

6518


In [24]:
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 broadcast
1 communiti
2 decid
3 licenc
4 awar
5 defam
6 wit
7 call
8 infrastructur
9 protect
10 summit


### **Gensim filter_extremes**

Filter out tokens that appear in **less than 15 documents** (absolute number) or **more than 0.5 documents** (fraction of total corpus size, not absolute number).

After the above two steps, keep only the first 100.000 most frequent tokens.

In [25]:
dictionary.filter_extremes(no_below=10, no_above=0.5, keep_n=100000)

print(len(dictionary.iteritems())) # This tells you how many tokens remain after filtering.

1049


### **Gensim doc2bow**

For each document we create a dictionary reporting how many
words and how many times those words appear. Save this to ‘bow_corpus’, then check our selected document earlier.

In [26]:
# Use dictionary.doc2bow(doc) inside a list comprehension to create 'bow_corpus'
# Each entry will show how many times each token (by ID) appears in a document
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

docnr = 4310

print(bow_corpus[docnr])
print('length of corpus: ', len(bow_corpus))

[(99, 1), (152, 1), (182, 1), (376, 1), (505, 1)]
length of corpus:  10000


Preview Bag Of Words for our sample preprocessed document.

In [27]:
bow_doc_4310 = bow_corpus[docnr]

for i in range(len(bow_doc_4310)):
    print("Word {} (\"{}\") appears {} time.".format(bow_doc_4310[i][0], 
                                                     dictionary[bow_doc_4310[i][0]], 
                                                     bow_doc_4310[i][1]))

Word 99 ("govt") appears 1 time.
Word 152 ("group") appears 1 time.
Word 182 ("vote") appears 1 time.
Word 376 ("local") appears 1 time.
Word 505 ("want") appears 1 time.


#### **TF-IDF**

Create tf-idf model object using models.TfidfModel on `bow_corpus` and save it to `tfidf`, 

then apply transformation to the entire corpus and call it `corpus_tfidf`. Finally we preview TF-IDF scores for our first document.



In [28]:
from gensim import corpora, models

tfidf = models.TfidfModel(bow_corpus)

In [29]:
# Apply the model to the entire corpus to get TF-IDF weights
# This works just like a transformation: tfidf[bow_corpus]
# Save the result to a new variable called 'corpus_tfidf'
corpus_tfidf = tfidf[bow_corpus]

In [30]:
# We use a loop to go through the tf-idf-transformed corpus using pprint for a nice display (only the first doc)
from pprint import pprint

for doc in corpus_tfidf:
    pprint(doc)
    break

[(0, 0.6441604901095495), (1, 0.7648903601051755)]


Each tuple is: (token_id, tfidf_score)

So in this document
- word with ID 0 has a TF-IDF score of 0.64 
- word with ID 1 has a TF-IDF score of 0.76

These values reflect:
- How important each word is in this specific document
- Relative to how common the word is across the entire corpus



---

### Running LDA using Bag of Words

Train our lda model using gensim.models.LdaMulticore and save it to ‘lda_model’

In [31]:
# use following parameters in: 'corpus', 'num_topics', 'id2word', 'passes' and 'workers'
lda_model = gensim.models.LdaMulticore(bow_corpus, num_topics=10, 
                                       id2word=dictionary, passes=2, 
                                       workers=2)

# corpus:       the input data — each document in BoW format
# num_topics:   how many topics the model should learn
# id2word:      maps word IDs back to actual words (used for display)
# passes:       how many times to go through the entire corpus
# workers:      how many CPU cores to use for faster training

/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [32]:
# Use lda_model.print_topics(-1) to get all topics with their keywords
# Loop through the output and print each topic number and its top words

for idx, topic in lda_model.print_topics(-1):
    print('Topic: {} \nWords: {}'.format(idx, topic))

Topic: 0 
Words: 0.018*"urg" + 0.012*"crash" + 0.012*"face" + 0.011*"train" + 0.010*"council" + 0.010*"court" + 0.010*"take" + 0.009*"accid" + 0.009*"close" + 0.009*"shark"
Topic: 1 
Words: 0.017*"govt" + 0.013*"kill" + 0.012*"hous" + 0.011*"worker" + 0.010*"investig" + 0.009*"miss" + 0.009*"anti" + 0.009*"attack" + 0.009*"plan" + 0.009*"journalist"
Topic: 2 
Words: 0.022*"say" + 0.017*"fund" + 0.014*"anti" + 0.013*"plan" + 0.011*"baghdad" + 0.010*"govt" + 0.009*"week" + 0.009*"rise" + 0.008*"iraq" + 0.008*"miss"
Topic: 3 
Words: 0.037*"polic" + 0.013*"concern" + 0.012*"sar" + 0.012*"australia" + 0.011*"troop" + 0.010*"death" + 0.010*"woman" + 0.010*"hospit" + 0.010*"iraq" + 0.009*"home"
Topic: 4 
Words: 0.023*"charg" + 0.021*"iraqi" + 0.020*"govt" + 0.020*"baghdad" + 0.013*"world" + 0.012*"year" + 0.010*"continu" + 0.010*"lead" + 0.010*"murder" + 0.009*"win"
Topic: 5 
Words: 0.037*"iraq" + 0.023*"polic" + 0.016*"say" + 0.011*"welcom" + 0.011*"troop" + 0.010*"howard" + 0.009*"industri"

Cool! Can you distinguish different topics using the words in each topic and their corresponding weights?

## Visualization

There is a nice way to visualize the LDA model you built using the package pyLDAvis:

In [43]:
!pip install pyLDAvis --quiet
!pip install pyLDAvis.gensim --quiet 

ERROR: Could not find a version that satisfies the requirement pyLDAvis.gensim (from versions: none)
ERROR: No matching distribution found for pyLDAvis.gensim


In [34]:
%matplotlib inline
import pyLDAvis
import pyLDAvis.gensim_models
import gensim

vis = pyLDAvis.gensim_models.prepare(lda_model, bow_corpus, dictionary)
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)

**Allocating topics to documents**

In [35]:
processed_docs[4310]

['ratepay', 'group', 'want', 'compulsori', 'local', 'govt', 'vote']

In [36]:
for index, score in sorted(lda_model[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model.print_topic(index, 10)))


Score: 0.8499656915664673	 
Topic: 0.037*"iraq" + 0.023*"polic" + 0.016*"say" + 0.011*"welcom" + 0.011*"troop" + 0.010*"howard" + 0.009*"industri" + 0.009*"post" + 0.009*"saddam" + 0.008*"want"

Score: 0.016673004254698753	 
Topic: 0.019*"council" + 0.016*"sar" + 0.014*"north" + 0.013*"elect" + 0.012*"iraq" + 0.011*"china" + 0.011*"korea" + 0.010*"virus" + 0.009*"court" + 0.009*"return"

Score: 0.016672585159540176	 
Topic: 0.017*"govt" + 0.013*"kill" + 0.012*"hous" + 0.011*"worker" + 0.010*"investig" + 0.009*"miss" + 0.009*"anti" + 0.009*"attack" + 0.009*"plan" + 0.009*"journalist"

Score: 0.01667184568941593	 
Topic: 0.023*"charg" + 0.021*"iraqi" + 0.020*"govt" + 0.020*"baghdad" + 0.013*"world" + 0.012*"year" + 0.010*"continu" + 0.010*"lead" + 0.010*"murder" + 0.009*"win"

Score: 0.016670644283294678	 
Topic: 0.017*"plan" + 0.016*"claim" + 0.011*"state" + 0.011*"protest" + 0.009*"land" + 0.009*"battl" + 0.009*"govt" + 0.009*"start" + 0.009*"consid" + 0.008*"meet"

Score: 0.016670530

## Testing model on unseen document

In [37]:
unseen_document = 'Payment shock awaits US home buyers as mortgage rates climb'
bow_vector = dictionary.doc2bow(preprocess(unseen_document))

print('bow_vector: ', bow_vector)

for index, score in sorted(lda_model[bow_vector], key=lambda tup: -1*tup[1]):
    print("Score: {}\t Topic: {}".format(score, lda_model.print_topic(index, 5)))

bow_vector:  [(64, 1), (65, 1), (429, 1), (535, 1), (956, 1)]
Score: 0.8499414324760437	 Topic: 0.018*"urg" + 0.012*"crash" + 0.012*"face" + 0.011*"train" + 0.010*"council"
Score: 0.016677584499120712	 Topic: 0.024*"report" + 0.023*"iraqi" + 0.013*"shoot" + 0.011*"sar" + 0.011*"kill"
Score: 0.016675811260938644	 Topic: 0.017*"govt" + 0.013*"kill" + 0.012*"hous" + 0.011*"worker" + 0.010*"investig"
Score: 0.016673561185598373	 Topic: 0.022*"say" + 0.017*"fund" + 0.014*"anti" + 0.013*"plan" + 0.011*"baghdad"
Score: 0.01667330041527748	 Topic: 0.037*"polic" + 0.013*"concern" + 0.012*"sar" + 0.012*"australia" + 0.011*"troop"
Score: 0.016672946512699127	 Topic: 0.037*"iraq" + 0.023*"polic" + 0.016*"say" + 0.011*"welcom" + 0.011*"troop"
Score: 0.016672834753990173	 Topic: 0.019*"council" + 0.016*"sar" + 0.014*"north" + 0.013*"elect" + 0.012*"iraq"
Score: 0.016672424972057343	 Topic: 0.055*"iraq" + 0.033*"baghdad" + 0.019*"bomb" + 0.017*"plan" + 0.014*"iraqi"
Score: 0.016670558601617813	 Topic

Our test document has the highest probability to be part of the topic on the top.

## Exercise: 7.1 

Notice that the above dataset is biased towards australian news. 

Run the LDA model on a unseen document and compare the two. You can search Kaggle, NLTK or any other source of your choice.


In [38]:
lda_model_tfidf = gensim.models.LdaMulticore(corpus_tfidf, num_topics=10, id2word=dictionary, passes=2, workers=4)

/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/johannuchelwildt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+,

In [39]:
for idx, topic in lda_model_tfidf.print_topics(-1):
    print('Topic: {} Word: {}'.format(idx, topic))

Topic: 0 Word: 0.013*"iraq" + 0.012*"troop" + 0.011*"win" + 0.011*"dead" + 0.010*"industri" + 0.007*"face" + 0.007*"deni" + 0.007*"accus" + 0.007*"injur" + 0.007*"polic"
Topic: 1 Word: 0.015*"kill" + 0.012*"protest" + 0.011*"open" + 0.011*"iraq" + 0.010*"crash" + 0.010*"world" + 0.009*"turkey" + 0.009*"british" + 0.008*"helicopt" + 0.008*"leader"
Topic: 2 Word: 0.015*"protest" + 0.015*"anti" + 0.014*"polic" + 0.010*"govt" + 0.008*"call" + 0.008*"trial" + 0.008*"send" + 0.008*"turn" + 0.008*"iraq" + 0.007*"council"
Topic: 3 Word: 0.020*"plan" + 0.012*"vote" + 0.011*"gold" + 0.011*"baghdad" + 0.010*"urg" + 0.010*"council" + 0.008*"raid" + 0.007*"coast" + 0.007*"coalit" + 0.007*"say"
Topic: 4 Word: 0.018*"warn" + 0.008*"iraq" + 0.008*"mayor" + 0.008*"fear" + 0.007*"launch" + 0.007*"say" + 0.007*"adelaid" + 0.007*"israel" + 0.007*"predict" + 0.007*"miss"
Topic: 5 Word: 0.011*"iraq" + 0.010*"labor" + 0.010*"minist" + 0.009*"question" + 0.009*"return" + 0.009*"bomber" + 0.008*"govt" + 0.008*

In [40]:
processed_docs[4310]

['ratepay', 'group', 'want', 'compulsori', 'local', 'govt', 'vote']

In [41]:
for index, score in sorted(lda_model_tfidf[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model_tfidf.print_topic(index, 10)))


Score: 0.6084645986557007	 
Topic: 0.011*"iraq" + 0.010*"labor" + 0.010*"minist" + 0.009*"question" + 0.009*"return" + 0.009*"bomber" + 0.008*"govt" + 0.008*"polic" + 0.008*"tour" + 0.008*"rule"

Score: 0.25816404819488525	 
Topic: 0.020*"plan" + 0.012*"vote" + 0.011*"gold" + 0.011*"baghdad" + 0.010*"urg" + 0.010*"council" + 0.008*"raid" + 0.007*"coast" + 0.007*"coalit" + 0.007*"say"

Score: 0.01667328178882599	 
Topic: 0.018*"iraqi" + 0.012*"iraq" + 0.009*"govt" + 0.009*"missil" + 0.008*"surrend" + 0.008*"say" + 0.008*"saddam" + 0.008*"start" + 0.007*"battl" + 0.007*"rain"

Score: 0.01667191833257675	 
Topic: 0.015*"protest" + 0.015*"anti" + 0.014*"polic" + 0.010*"govt" + 0.008*"call" + 0.008*"trial" + 0.008*"send" + 0.008*"turn" + 0.008*"iraq" + 0.007*"council"

Score: 0.016671504825353622	 
Topic: 0.017*"lead" + 0.012*"death" + 0.011*"drought" + 0.010*"toll" + 0.009*"claim" + 0.009*"announc" + 0.008*"die" + 0.007*"take" + 0.007*"record" + 0.007*"hop"

Score: 0.01667133718729019	 
T

In [42]:
def generate_nlp_tool_comparison():
    return """
### Gensim vs NLTK Comparison

| Feature           | Gensim                                               | NLTK                                                       |
|-------------------|------------------------------------------------------|-------------------------------------------------------------|
| **Focus**         | Topic modeling, vector space models, scalable NLP    | Educational, linguistically rich NLP tools                  |
| **Use case**      | LDA, word2vec, doc2vec, semantic modeling            | POS tagging, parsing, named entity recognition              |
| **Speed/Scale**   | Built for large-scale text processing                | Slower, not optimized for big data                          |
| **Industry adoption** | Popular in production for unsupervised topic modeling and word embeddings | Common in academia or for prototyping      |
"""

# To display it in a Jupyter Notebook:
from IPython.display import Markdown
display(Markdown(generate_nlp_tool_comparison()))



### Gensim vs NLTK Comparison

| Feature           | Gensim                                               | NLTK                                                       |
|-------------------|------------------------------------------------------|-------------------------------------------------------------|
| **Focus**         | Topic modeling, vector space models, scalable NLP    | Educational, linguistically rich NLP tools                  |
| **Use case**      | LDA, word2vec, doc2vec, semantic modeling            | POS tagging, parsing, named entity recognition              |
| **Speed/Scale**   | Built for large-scale text processing                | Slower, not optimized for big data                          |
| **Industry adoption** | Popular in production for unsupervised topic modeling and word embeddings | Common in academia or for prototyping      |


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=a62772d5-473c-4698-8b17-a8c47ba190c5' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>